# 01 - Mongo JSON -> relational tables (Parquet)

**Goal:** turn the stitched daily documents into flat tables that are easy to join, clean and train on.

- **Read-only.** Nothing is written to Mongo. Only local data helpers are imported; no API extractors are run.
- **Input:** `SOURCE = "json"` reads `data/pipeline/out/state_signals_daily_*.json`. `SOURCE = "mongo"` reads the live collection.
- **Output:** one Parquet file per table in `data/pipeline/out/tables/` (git-ignored).
- **Run order:** top to bottom. The checks cell blocks the save cell if a real problem is found.

| Table | One row per |
|---|---|
| `state`, `state_day` | state / state + day |
| `source_status` | state + day + state source |
| `national_source_status` | country + day + national source |
| `weather` | state + day |
| `news_article`, `trend`, `holiday`, `reddit_post`, `event` | item inside a state + day |
| `attention_article`, `sports_event`, `youtube_video` | item for a day (national, stored once, no state) |

Processes one reference day at a time. `RUN_DATE=None` selects the latest saved day; set `RUN_DATE="YYYY-MM-DD"` to rebuild an older day. Outputs are partitioned under `tables/date=YYYY-MM-DD/`. Legacy embedded national JSON and new shared national snapshots are both supported.

National snapshots are converted independently, with one `national_source_status` row per country/day/source. State documents are never expanded with national content.


Install dependencies once with `python -m pip install -r data/pipeline/notebooks/requirements.txt`.


In [2]:
import json
import re
import sys
import os
from collections import Counter
from pathlib import Path

import pandas as pd

# ---- settings -------------------------------------------------------------
SOURCE = "json"      # "json" -> out/state_signals_daily_*.json   |   "mongo" -> live collection (read-only)
RUN_DATE = os.environ.get("SIGNAL_RUN_DATE")  # YYYY-MM-DD; None selects latest day

# ---- paths (found by walking up from wherever the notebook is run) ----------
_here = Path.cwd().resolve()
ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "pipeline" / "run_extract.py").exists())
PIPELINE_DIR = ROOT / "data" / "pipeline"
JSON_DIR = PIPELINE_DIR / "out"
OUT_DIR = PIPELINE_DIR / "out" / "tables"

ALL_SOURCES = ["weather", "news", "trends", "calendar", "social", "events"]
NATIONAL_SOURCES = ["attention", "sports", "youtube", "calendar"]

if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from validate import check_snapshot_pair


## 1. Load the documents

In [3]:
def load_docs():
    """Return one stitched doc per (state_code, date). If a pair appears twice, keep the newest."""
    if SOURCE == "json":
        paths = sorted(JSON_DIR.glob("state_signals_daily_*.json"))
        if not paths:
            raise FileNotFoundError(f"No state_signals_daily_*.json in {JSON_DIR}")
        available = {p.stem.removeprefix("state_signals_daily_"): p for p in paths}
        selected = RUN_DATE or max(available)
        if selected not in available:
            raise FileNotFoundError(f"No state snapshot for {selected}")
        paths = [available[selected]]
        docs = json.loads(paths[0].read_text())
        national_path = JSON_DIR / f"national_signals_daily_{selected}.json"
        national = json.loads(national_path.read_text()) if national_path.exists() else None
        print(f"Read {len(docs)} docs from {len(paths)} file(s)")
    elif SOURCE == "mongo":
        import os
        from dotenv import load_dotenv
        from pymongo import MongoClient

        load_dotenv(PIPELINE_DIR / ".env", override=False)
        load_dotenv(ROOT / "src" / "backend" / ".env", override=False)
        uri = os.getenv("MONGODB_URI") or os.getenv("MONGO_URI")
        if not uri:
            raise RuntimeError("Set MONGODB_URI in data/pipeline/.env or src/backend/.env")
        query = {}
        client = MongoClient(uri, serverSelectionTimeoutMS=5000)   # fail fast if unreachable
        try:
            coll = client[os.getenv("MONGODB_DB_NAME", "wavelength")]["state_signals_daily"]
            selected = RUN_DATE
            if selected is None:
                newest = coll.find_one(query, {"date": 1}, sort=[("date", -1)])
                if newest is None:
                    raise ValueError("No state snapshots matched the query")
                selected = newest["date"]
            query["date"] = selected
            docs = list(coll.find(query, {"_id": 0}))
            national = client[os.getenv("MONGODB_DB_NAME", "wavelength")]["national_signals_daily"].find_one(
                {"date": selected, "country_code": "US"}, {"_id": 0})
        finally:
            client.close()
        print(f"Read {len(docs)} docs from Mongo")
    else:
        raise ValueError("SOURCE must be 'json' or 'mongo'")

    check_snapshot_pair(docs, national)
    latest = {}
    for doc in docs:
        key = (doc["state_code"], doc["date"])
        if key not in latest or (doc.get("stitched_at") or "") >= (latest[key].get("stitched_at") or ""):
            latest[key] = doc
    if len(latest) < len(docs):
        print(f"Dropped {len(docs) - len(latest)} older duplicate doc(s)")
    return list(latest.values()), [national] if national is not None else []


docs, national_docs = load_docs()
if not docs:
    raise ValueError("No documents to convert")
REFERENCE_DATE = docs[0]["date"]
OUT_DIR = PIPELINE_DIR / "out" / "tables" / f"date={REFERENCE_DATE}"


Read 50 docs from 1 file(s)


## 2. Flatten

Collect rows in one pass. National items and explicitly national holidays are stored once per reference day. Conflicting copies of attention, sports or YouTube block saving. Legacy and shared snapshots are both supported.


In [4]:
COLUMNS = {
    "state":             ["state_code", "state_name", "census_region", "census_division"],
    "state_day":         ["state_code", "date", "stitched_at"],
    "source_status":     ["state_code", "date", "source", "status", "error", "fetched_at", "geo_scope", "geo_confidence"],
    "national_source_status": ["country_code", "date", "source", "status", "error", "fetched_at", "geo_scope", "geo_confidence"],
    "weather":           ["state_code", "date", "temperature_c", "temperature_max_c", "temperature_min_c",
                          "precipitation_mm", "weather_code", "condition"],
    "news_article":      ["state_code", "date", "position", "title", "url", "domain", "source_country", "language", "published_at"],
    "trend":             ["state_code", "date", "position", "rank", "keyword", "volume_min", "news_headline"],
    "holiday":           ["state_code", "date", "name", "holiday_date", "type", "state_specific", "is_today", "is_national", "observance_note"],
    "reddit_post":       ["state_code", "date", "position", "subreddit", "subreddit_confirmed", "title", "score", "url", "created_at"],
    "event":             ["state_code", "date", "position", "name", "url", "start_date", "venue_name", "venue_city",
                          "venue_state", "classification", "genre", "price_min", "price_max"],
    "attention_article": ["date", "data_date", "position", "article"],
    "sports_event":      ["date", "league", "position", "home_team", "away_team", "venue_name", "venue_city",
                          "venue_state", "start_time", "status", "event_id"],
    "youtube_video":     ["date", "position", "video_id", "title", "channel_title", "category_id", "view_count", "published_at"],
}
ATTENTION_URL_DATE = re.compile(r"/(\d{4})/(\d{2})/(\d{2})/?$")   # attention's real data day is only in meta.url


def flatten(docs, national_docs=()):
    rows = {t: [] for t in COLUMNS}
    expected = Counter()                 # raw list sizes, compared with table sizes in the checks
    states, national_seen, national_conflicts = {}, {}, []
    national_holidays = set()

    def items(signal, key, table):
        lst = (signal or {}).get(key) or []
        expected[table] += len(lst)
        return lst

    def keep_national(day, src, payload):
        """True only the first time a (day, source) is seen; records copies that differ."""
        fp = json.dumps(payload, sort_keys=True)
        if (day, src) not in national_seen:
            national_seen[(day, src)] = fp
            return True
        if national_seen[(day, src)] != fp:
            national_conflicts.append((day, src))
        return False

    def add_holidays(sc, day, cal):
        # holidays_today is a subset of holidays_this_month -> one table with an is_today flag
        today_list = cal.get("holidays_today") or []
        today_keys = {(h.get("name"), h.get("date"), h.get("type")) for h in today_list}
        merged = {}
        for h in (cal.get("holidays_this_month") or []) + today_list:
            merged[(h.get("name"), h.get("date"), h.get("type"))] = h
        for key, h in merged.items():
            is_national = True if sc is None else h.get("is_national")
            if is_national is True:
                shared_key = (day, *key)
                if shared_key in national_holidays:
                    continue
                national_holidays.add(shared_key)
            expected["holiday"] += 1
            rows["holiday"].append((None if is_national is True else sc, day, h.get("name"), h.get("date"),
                                    h.get("type"), h.get("state_specific"), key in today_keys, is_national, h.get("observance_note")))


    def add_national(day, source, signal, failure=None):
        if not keep_national(day, source, {"signal": signal, "failure": failure}):
            return
        signal = signal or {}
        meta = signal.get("meta") or {}
        rows["national_source_status"].append((
            "US", day, source, meta.get("status") or ("error" if failure else "missing"),
            meta.get("error") or (failure.get("error") if failure else None),
            meta.get("fetched_at") or (failure.get("fetched_at") if failure else None), "national", "exact"))
        if source == "calendar":
            add_holidays(None, day, signal)
        elif source == "attention":
            match = ATTENTION_URL_DATE.search(meta.get("url") or "")
            data_day = "-".join(match.groups()) if match else None
            for i, title in enumerate(items(signal, "top_articles", "attention_article"), 1):
                rows["attention_article"].append((day, data_day, i, title))
        elif source == "sports":
            for i, event in enumerate(items(signal, "events", "sports_event"), 1):
                rows["sports_event"].append((day, event.get("league") or signal.get("league"), i, event.get("home_team"), event.get("away_team"),
                                              event.get("venue_name"), event.get("venue_city"), event.get("venue_state"),
                                              event.get("start_time"), event.get("status"), event.get("event_id")))
        elif source == "youtube":
            for i, video in enumerate(items(signal, "videos", "youtube_video"), 1):
                rows["youtube_video"].append((day, i, video.get("video_id"), video.get("title"), video.get("channel_title"),
                                               video.get("category_id"), video.get("view_count"), video.get("published_at")))

    for doc in docs:
        sc, day = doc["state_code"], doc["date"]
        sig = doc.get("signals") or {}
        states[sc] = (sc, doc.get("state_name"), doc.get("census_region"), doc.get("census_division"))
        rows["state_day"].append((sc, day, doc.get("stitched_at")))

        # one status row per source, including sources that are missing from the doc
        for src in ALL_SOURCES:
            s = sig.get(src)
            if not isinstance(s, dict):
                failure = (doc.get("source_errors") or {}).get(src)
                rows["source_status"].append((sc, day, src, "error" if failure else "missing",
                                              failure.get("error") if failure else None,
                                              failure.get("fetched_at") if failure else None, None, None))
                continue
            m, geo = s.get("meta") or {}, s.get("geo") or {}
            rows["source_status"].append((sc, day, src, m.get("status"), m.get("error"), m.get("fetched_at"),
                                          geo.get("scope") or s.get("scope") or "state", geo.get("confidence")))

        w = sig.get("weather")
        if isinstance(w, dict):
            rows["weather"].append((sc, day, w.get("temperature_c"), w.get("temperature_max_c"), w.get("temperature_min_c"),
                                    w.get("precipitation_mm"), w.get("weather_code"), w.get("condition")))

        for i, a in enumerate(items(sig.get("news"), "articles", "news_article"), 1):
            rows["news_article"].append((sc, day, i, a.get("title"), a.get("url"), a.get("domain"),
                                         a.get("source_country"), a.get("language"), a.get("seendate")))

        for i, t in enumerate(items(sig.get("trends"), "trends", "trend"), 1):
            rows["trend"].append((sc, day, i, t.get("rank"), t.get("keyword"), t.get("volume_min"), t.get("news_headline")))

        cal = sig.get("calendar") or {}
        add_holidays(sc, day, cal)
        shared_holidays = [h for h in (cal.get("holidays_this_month") or []) if h.get("is_national") is True]
        if shared_holidays:
            add_national(day, "calendar", {**cal, "holidays_this_month": shared_holidays,
                                          "holidays_today": [h for h in shared_holidays if h.get("date") == day]})

        soc = sig.get("social") or {}
        for i, p in enumerate(items(soc, "posts", "reddit_post"), 1):
            rows["reddit_post"].append((sc, day, i, soc.get("subreddit"), soc.get("subreddit_confirmed"), p.get("title"),
                                        p.get("score"), p.get("url"), p.get("created_utc")))

        for i, e in enumerate(items(sig.get("events"), "events", "event"), 1):
            rows["event"].append((sc, day, i, e.get("name"), e.get("url"), e.get("start_date"), e.get("venue_name"),
                                  e.get("venue_city"), e.get("venue_state"), e.get("classification"), e.get("genre"),
                                  e.get("price_min"), e.get("price_max")))

        # Legacy input only: extract embedded national data directly, without reattaching it.
        for src in ("attention", "sports", "youtube"):
            if src in sig:
                add_national(day, src, sig[src], (doc.get("source_errors") or {}).get(src))

    for national in national_docs:
        day = national["date"]
        for src in NATIONAL_SOURCES:
            add_national(day, src, (national.get("signals") or {}).get(src),
                         (national.get("source_errors") or {}).get(src))
    for day in {doc["date"] for doc in docs}:
        for src in NATIONAL_SOURCES:
            if (day, src) not in national_seen:
                add_national(day, src, None)

    rows["state"] = sorted(states.values())
    return rows, expected, national_conflicts


rows, expected, national_conflicts = flatten(docs, national_docs)
print({t: len(r) for t, r in rows.items()})


{'state': 50, 'state_day': 50, 'source_status': 300, 'national_source_status': 4, 'weather': 50, 'news_article': 0, 'trend': 0, 'holiday': 0, 'reddit_post': 960, 'event': 0, 'attention_article': 0, 'sports_event': 84, 'youtube_video': 0}


## 3. Build typed DataFrames

In [5]:
DAY_COLS = ["date", "holiday_date", "data_date", "start_date"]                                   # calendar days
TIME_COLS = ["stitched_at", "fetched_at", "published_at", "start_time"]  # ISO timestamps -> UTC
INT_COLS = ["position", "rank", "volume_min", "score", "weather_code", "view_count"]
FLOAT_COLS = ["temperature_c", "temperature_max_c", "temperature_min_c", "precipitation_mm", "price_min", "price_max"]
BOOL_COLS = ["state_specific", "is_today", "subreddit_confirmed", "is_national"]
CAT_COLS = ["state_code", "source", "status", "geo_scope", "geo_confidence", "census_region", "census_division"]


def build_tables(rows):
    tables = {}
    conversion_failures = []
    for name, data in rows.items():
        df = pd.DataFrame(data, columns=COLUMNS[name])
        if name == "reddit_post":   # created_utc is epoch seconds
            before_epoch = df["created_at"].copy()
            df["created_at"] = pd.to_datetime(df["created_at"], unit="s", utc=True, errors="coerce")
            lost = int((before_epoch.notna() & df["created_at"].isna()).sum())
            if lost:
                conversion_failures.append((name, "created_at", lost))
        for c in df.columns:
            before = df[c].copy()
            if c in DAY_COLS:
                df[c] = pd.to_datetime(df[c], format="%Y-%m-%d", errors="coerce")
            elif c in TIME_COLS or (c == "created_at" and name != "reddit_post"):
                df[c] = pd.to_datetime(df[c], format="ISO8601", utc=True, errors="coerce")
            elif c in INT_COLS:
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
            elif c in FLOAT_COLS:
                df[c] = pd.to_numeric(df[c], errors="coerce")
            elif c in BOOL_COLS:
                df[c] = df[c].astype("boolean")
            elif c in CAT_COLS:
                df[c] = df[c].astype("category")
            if c in DAY_COLS + TIME_COLS + INT_COLS + FLOAT_COLS:
                lost = int((before.notna() & df[c].isna()).sum())
                if lost:
                    conversion_failures.append((name, c, lost))
        tables[name] = df
    if conversion_failures:
        raise ValueError(f"Values could not be converted (table, column, rows): {conversion_failures}")
    return tables


tables = build_tables(rows)
pd.DataFrame({"rows": {t: len(df) for t, df in tables.items()}, "columns": {t: df.shape[1] for t, df in tables.items()}})


,rows,columns
state,50,4
state_day,50,3
source_status,300,8
national_source_status,4,8
weather,50,8
news_article,0,9
trend,0,7
holiday,0,9
reddit_post,960,9
event,0,13


## 4. Checks (bug detectors)

`error` checks block the save. `warn` checks are shown but do not block (for example a partial run with fewer than 50 states).


In [6]:
PRIMARY_KEYS = {
    "state": ["state_code"],
    "state_day": ["state_code", "date"],
    "source_status": ["state_code", "date", "source"],
    "national_source_status": ["country_code", "date", "source"],
    "weather": ["state_code", "date"],
    "news_article": ["state_code", "date", "position"],
    "trend": ["state_code", "date", "position"],
    "holiday": ["state_code", "date", "name", "holiday_date", "type"],
    "reddit_post": ["state_code", "date", "position"],
    "event": ["state_code", "date", "position"],
    "attention_article": ["date", "position"],
    "sports_event": ["date", "position"],
    "youtube_video": ["date", "position"],
}
STATE_LEVEL = ["source_status", "weather", "news_article", "trend", "holiday", "reddit_post", "event"]
NATIONAL = ["attention_article", "sports_event", "youtube_video", "national_source_status"]


def run_checks(t, expected, national_conflicts):
    out = []
    add = lambda name, level, ok, detail="": out.append((name, level, bool(ok), detail))

    sd = t["state_day"]
    add("state_day: reference timestamps present", "error", sd["stitched_at"].notna().all())
    spine = pd.MultiIndex.from_frame(sd[["state_code", "date"]].astype({"state_code": "object"}))
    days = set(sd["date"])

    for name, pk in PRIMARY_KEYS.items():
        required = [c for c in pk if not (name == "holiday" and c == "state_code")]
        add(f"{name}: required key values present", "error", t[name][required].notna().all().all())
        dup = int(t[name].duplicated(pk).sum())
        add(f"{name}: key {pk} is unique", "error", dup == 0, f"{dup} duplicate rows")

    for name in STATE_LEVEL:
        state_rows = t[name].loc[t[name]["state_code"].notna()]
        idx = pd.MultiIndex.from_frame(state_rows[["state_code", "date"]].astype({"state_code": "object"}))
        orphans = int((~idx.isin(spine)).sum())
        add(f"{name}: every row has a state_day row", "error", orphans == 0, f"{orphans} orphan rows")
    for name in NATIONAL:
        orphans = int((~t[name]["date"].isin(days)).sum())
        add(f"{name}: every row has a matching day", "error", orphans == 0, f"{orphans} orphan rows")

    add("source_status: one row per state_day x source", "error", len(t["source_status"]) == len(sd) * len(ALL_SOURCES),
        f"{len(t['source_status'])} vs {len(sd)} x {len(ALL_SOURCES)}")
    add("national_source_status: one row per day x source", "error",
        len(t["national_source_status"]) == len(days) * len(NATIONAL_SOURCES))
    for name, n in expected.items():
        add(f"{name}: rows match raw list sizes", "error", len(t[name]) == n, f"table {len(t[name])} vs raw {n}")
    add("national sources identical across states", "error", not national_conflicts, f"conflicts: {national_conflicts[:3]}")

    hol = t["holiday"]
    national_holidays = hol["state_code"].isna()
    add("national holidays: explicit scope and matching day", "error",
        hol.loc[national_holidays, "is_national"].fillna(False).all()
        and hol.loc[national_holidays, "date"].isin(days).all())
    for table, cols in {"weather": ["precipitation_mm"], "event": ["price_min", "price_max"],
                        "trend": ["volume_min"], "youtube_video": ["view_count"]}.items():
        add(f"{table}: nonnegative values", "error", (t[table][cols].ge(0) | t[table][cols].isna()).all().all())
    prices = t["event"]
    known = prices["price_min"].notna() & prices["price_max"].notna()
    add("event: price order valid", "error", prices.loc[known, "price_min"].le(prices.loc[known, "price_max"]).all())
    add("source_status: no source notices", "warn", t["source_status"]["error"].isna().all(),
        str(t["source_status"].loc[t["source_status"]["error"].notna(), ["source", "error"]].to_dict("records")))
    n_states, n_days = len(t["state"]), sd["date"].nunique()
    add("state_day rows == states x days", "warn", len(sd) == n_states * n_days, f"{len(sd)} vs {n_states} x {n_days}")
    add("50 states present", "warn", n_states == 50, f"{n_states} states")
    ok_share = (t["source_status"]["status"].astype(str) == "ok").mean()
    add("source_status: all sources ok", "warn", ok_share == 1, f"{ok_share:.0%} ok; non-ok: "
        + str(t["source_status"].loc[t["source_status"]["status"].astype(str) != "ok", "source"].astype(str).value_counts().to_dict()))
    national_status = t["national_source_status"]
    add("national_source_status: no source notices", "warn", national_status["error"].isna().all(),
        str(national_status.loc[national_status["error"].notna(), ["source", "error"]].to_dict("records")))
    add("national_source_status: sources ok or empty", "warn",
        national_status["status"].isin(["ok", "empty"]).all(),
        str(national_status.loc[~national_status["status"].isin(["ok", "empty"]), ["source", "status"]].to_dict("records")))
    return pd.DataFrame(out, columns=["check", "level", "ok", "detail"])


report = run_checks(tables, expected, national_conflicts)
failed = report[~report["ok"]]
print(f"{len(report) - len(failed)}/{len(report)} checks passed")
failed if len(failed) else "all checks passed"


56/60 checks passed


,check,level,ok,detail
54,source_status: no source notices,warn,False,"[{'source': 'news', 'error': 'NEWSDATA_API_KEY..."
57,source_status: all sources ok,warn,False,"33% ok; non-ok: {'news': 50, 'trends': 50, 'ca..."
58,national_source_status: no source notices,warn,False,"[{'source': 'attention', 'error': 'WIKIMEDIA_C..."
59,national_source_status: sources ok or empty,warn,False,"[{'source': 'attention', 'status': 'error'}, {..."


In [7]:
# Null share per column (only columns that have nulls)
null_report = pd.DataFrame(
    [(t, c, round(df[c].isna().mean() * 100, 1)) for t, df in tables.items() if len(df) for c in df.columns if df[c].isna().any()],
    columns=["table", "column", "null_pct"],
).sort_values("null_pct", ascending=False)
null_report


,table,column,null_pct
1,source_status,geo_scope,67.3
2,source_status,geo_confidence,67.3
0,source_status,error,32.7
3,national_source_status,error,25.0
4,sports_event,venue_state,9.5


## 5. Save to Parquet

In [8]:
report = run_checks(tables, expected, national_conflicts)
blocking = report[(report["level"] == "error") & ~report["ok"]]
if len(blocking):
    raise RuntimeError(f"Fix these before saving:\n{blocking[['check', 'detail']].to_string(index=False)}")

OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in tables.items():
    df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)   # types (dates, nullable ints, categories) are preserved
print(f"Saved {len(tables)} tables to {OUT_DIR}")

quality = {
    "checks": report.to_dict("records"),
    "missing_values": null_report.to_dict("records"),
    "collection_windows": national_docs[0].get("windows") if national_docs else docs[0].get("windows"),
}
(OUT_DIR / "quality_report.json").write_text(json.dumps(quality, indent=2, default=str))


Saved 13 tables to /Users/mokshdagangrade/Desktop/MSML/MSML641/culture-fit-check/data/pipeline/out/tables/date=2026-10-20


31377

## 6. Join demo (re-read from disk)

Proves the saved files join cleanly: one row per state + day with weather and item counts.


In [9]:
saved = {p.stem: pd.read_parquet(p) for p in OUT_DIR.glob("*.parquet")}
key = ["state_code", "date"]
counts = (
    saved["news_article"].groupby(key, observed=True).size().rename("n_news").to_frame()
    .join(saved["trend"].groupby(key, observed=True).size().rename("n_trends"), how="outer")
    .join(saved["event"].groupby(key, observed=True).size().rename("n_events"), how="outer")
    .reset_index()
)
wide = (saved["state_day"]
        .merge(saved["state"], on="state_code")
        .merge(saved["weather"], on=key, how="left")
        .merge(counts, on=key, how="left"))
wide


,state_code,date,stitched_at,state_name,census_region,census_division,temperature_c,temperature_max_c,temperature_min_c,precipitation_mm,weather_code,condition,n_news,n_trends,n_events
0,AL,2026-10-20,2026-10-20 20:24:51.434379+00:00,Alabama,South,East South Central,25.7,25.7,5.9,0.0,0,clear sky,NaN,NaN,NaN
1,AK,2026-10-20,2026-10-20 20:24:57.383045+00:00,Alaska,West,Pacific,7.4,8.2,4.4,0.5,61,light rain,NaN,NaN,NaN
2,AZ,2026-10-20,2026-10-20 20:25:00.882808+00:00,Arizona,West,Mountain,32.6,33.4,20.6,0.0,1,mostly clear,NaN,NaN,NaN
3,AR,2026-10-20,2026-10-20 20:25:06.728815+00:00,Arkansas,South,West South Central,24.8,24.9,8.2,0.0,0,clear sky,NaN,NaN,NaN
4,CA,2026-10-20,2026-10-20 20:25:12.176223+00:00,California,West,Pacific,20.4,22.4,9.3,0.0,3,overcast,NaN,NaN,NaN
5,CO,2026-10-20,2026-10-20 20:25:19.918847+00:00,Colorado,West,Mountain,21.8,22.5,6.2,0.0,1,mostly clear,NaN,NaN,NaN
6,CT,2026-10-20,2026-10-20 20:25:23.890850+00:00,Connecticut,Northeast,New England,19.0,20.2,3.6,0.0,0,clear sky,NaN,NaN,NaN
7,DE,2026-10-20,2026-10-20 20:25:28.851026+00:00,Delaware,South,South Atlantic,20.1,20.2,6.9,0.0,0,clear sky,NaN,NaN,NaN
8,FL,2026-10-20,2026-10-20 20:25:31.783742+00:00,Florida,South,South Atlantic,23.6,23.6,8.7,0.0,0,clear sky,NaN,NaN,NaN
9,GA,2026-10-20,2026-10-20 20:25:35.680592+00:00,Georgia,South,South Atlantic,23.9,24.1,6.1,0.0,3,overcast,NaN,NaN,NaN
